In [38]:
import random

class Matrix:
    def __init__(self, data=None, dim=None, init_value= 0) -> None:
        self.data = data
        self.init_value = init_value
        if self.data == None:
            self.dim = dim
            #这里有一个未完成的初始化！！！  
        else:
            self.dim = (len(self.data), len(self.data[0]))
        
    def __repr__(self) -> str:
        return self.data
    
    def shape(self):
        return self.dim

    def reshape(self, newdim):
        if newdim[0] * newdim[1] != self.dim[0] * self.dim[1]:
            return "Error."
        
        tem = [0 for k in range(self.dim[0] * self.dim[1])]
        x = 0
        for i in range(self.dim[0]):
            for j in range(self.dim[1]):
                tem[x] = self.data[i][j]
                x += 1

        x = 0
        new_data = []
        for i in range(newdim[0]):
            new_data.append([])

        for i in range(newdim[0]):
            for j in range(newdim[1]):
                new_data[i].append(tem[x])
                x += 1

        return Matrix(data=new_data)

    def dot(self, other):
        if self.dim[1] != other.dim[0]:
            return "Error. 这两个矩阵不可点乘。"
        
        result = []
        for i in range(self.dim[0]):
            result.append([])

        for i in range(self.dim[0]):
            for j in range(other.dim[1]):
                sum = 0
                for k in range(self.dim[1]):
                    sum += self.data[i][k] * other.data[k][j]
                result[i].append(sum)

        return Matrix(data=result)

    def T(self):
        result = []
        for i in range(self.dim[1]):
            result.append([])
        
        for j in range(self.dim[1]):
            for i in range(self.dim[0]):
                result[j].append(self.data[i][j])

        return Matrix(data=result)

    def sum(self, axis=None):
        if axis == None:
            sum = 0
            for i in self.data:
                for j in i:
                    sum += j
            return Matrix(data=[[sum]])

        elif axis == 0:
            result = [[0 for x in range(self.dim[1])]]
            for i in range(self.dim[1]):
                for j in range(self.dim[0]):
                    result[0][i] += self.data[j][i]
            return Matrix(data=result)

        elif axis == 1:
            result = [[0] for x in range(self.dim[0])]
            for i in range(self.dim[0]):
                for j in range(self.dim[1]):
                    result[i][0] += self.data[i][j]
            return Matrix(data=result)

        else:
            return "Error. Please input 1 or 0."

    def copy(self):
        B = Matrix(data=self.data, dim=self.dim, init_value=self.init_value)
        return B

    def Kronecker_product(self, other):
        result = []
        for i in range(self.dim[0] * other.dim[0]):
            result.append([])

        for i in range(self.dim[0] * other.dim[0]):
            for j in range(self.dim[1] * other.dim[1]):
                result[i].append(\
                    self.data[i // other.dim[0]][j // other.dim[1]] \
                    * other.data[i % other.dim[0]][j % other.dim[1]])
        return Matrix(data=result)

    #######################################################
    def __getitem__(self, key):
        if isinstance(key, tuple):
            row, col = key
            if isinstance(row, slice) or isinstance(col, slice):
                # 如果行或列是切片对象，则进行切片操作
                if isinstance(row, slice):
                    rows = range(*row.indices(len(self.data)))
                else:
                    rows = [row]

                if isinstance(col, slice):
                    cols = range(*col.indices(len(self.data[0])))
                else:
                    cols = [col]

                # 使用切片后的行列范围构建新数组
                sliced_data = [[self.data[i][j] for j in cols] for i in rows]
                return sliced_data
            else:
                # 否则返回具体的元素
                return self.data[row][col]
        else:
            raise IndexError("Invalid index format for 2D array")

    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            row, col = key
            if isinstance(row, slice) or isinstance(col, slice):
                if isinstance(row, slice):
                    rows = range(*row.indices(len(self.data)))
                else:
                    rows = [row]

                if isinstance(col, slice):
                    cols = range(*col.indices(len(self.data[0])))
                else:
                    cols = [col]

                for x in range(len(rows)):
                    for y in range(len(cols)):
                        self.data[rows[x]][cols[y]] = value.data[x][y] 
                return
            else:
                self.data[row][col] = value
                return
        else:
            raise IndexError("Invalid index format for 2D array.")

    
    def __pow__(self, n):
        if self.dim[0] != self.dim[1]:
            return "Error. 此矩阵不是方阵"

        if n == 0:
            result = []
            for i in range(self.dim[0]):
                result.append([0 for x in range(self.dim[1])])
            for i in range(self.dim[0]):
                result[i][i] = 1
            return Matrix(data=result)

        half_pow = self ** (n//2)

        if n % 2 == 0:
            return half_pow.dot(half_pow)
        else:
            return self.dot(half_pow.dot(half_pow))

    def __add__(self, other):
        if not (self.dim[0] == other.dim[0] and \
            self.dim[1] == other.dim[1]):
            return "Error. The two matrix can't be added"

        result = []
        for i in range(self.dim[0]):
            result.append([])

        for i in range(self.dim[0]):
            for j in range(self.dim[1]):
                result[i].append(\
                    self.data[i][j] + other.data[i][j])

        return Matrix(data=result)

    def __sub__(self, other):
        if not (self.dim[0] == other.dim[0] and \
            self.dim[1] == other.dim[1]):
            return "Error. 这两个矩阵不可减"

        result = []
        for i in range(self.dim[0]):
            result.append([])

        for i in range(self.dim[0]):
            for j in range(self.dim[1]):
                result[i].append(\
                    self.data[i][j] - other.data[i][j])

        return Matrix(data=result)

    def __mul__(self, other):
        if not (self.dim[0] == other.dim[0] and \
            self.dim[1] == other.dim[1]):
            return "Error. 这两个矩阵不可乘。"

        result = []
        for i in range(self.dim[0]):
            result.append([])

        for i in range(self.dim[0]):
            for j in range(self.dim[1]):
                result[i].append(\
                    self.data[i][j] * other.data[i][j])

        return Matrix(data=result)

    def __len__(self):
        return self.dim[0] * self.dim[1]

    ##对齐？
    def __str__(self):
        result = "[["
        for i in range(self.dim[0]):
            for j in range(self.dim[1]):
                result = result + " " + str(self.data[i][j])
            result = result + "\n"
        result = result + "]]"
        return result

    
    def det(self):
        global det_num
        if self.dim[0] != self.dim[1]:
            return "Error. 此矩阵不是方阵."

        matrix = []
        for i in range(self.dim[0]):
            matrix.append([])
            for j in range(self.dim[1]):
                matrix[i].append(self.data[i][j])
        
        matrix = gauss(matrix)

        result = 1
        for i in range(len(matrix)):
            result *= matrix[i][i]
        result *= det_num

        if result == 0:
            return 0

        return result

    def inverse(self):
        if self.dim[0] != self.dim[1]:
            return "Error. 此矩阵不是方阵."

        matrix = []
        for i in range(self.dim[0]):
            matrix.append([])
            for j in range(self.dim[1]):
                matrix[i].append(self.data[i][j])
        
        for i in range(self.dim[0]):
            for j in range(self.dim[1]):
                matrix[i].append(0)
            matrix[i][i + self.dim[1]] = 1
        
        matrix = gauss(matrix)

        if is_all_zero(matrix[-1][:self.dim[1]]):
            return "Error. 此方阵不可逆。"

        result = []
        for i in range(len(matrix)):
            temp = [matrix[i][j] for j in range(self.dim[1], len(matrix[i]))]
            result.append(temp)

        return Matrix(data=result)

    def rank(self):
        matrix = []
        for i in range(self.dim[0]):
            matrix.append([])
            for j in range(self.dim[1]):
                matrix[i].append(self.data[i][j])
        matrix = gauss(matrix)
        for i in range(self.dim[0]):
            if is_all_zero(matrix[i]):
                break

        if i == self.dim[0] -1 and not is_all_zero(matrix[i]):
            return i + 1

        return i    

########################################################################################
########################################################################################
##Gauss:
def is_all_zero(lst):
    for i in lst:
        if i != 0:
            return False
    return True

def switch_line(array, k):
    global det_num
    for i in range(k, len(array)):
        if array[i][0] != 0:
            #print(f"before array : {array}, {det_num}")
            array[k], array[i] = array[i], array[k]
            if i != k:
                det_num *= -1
            #print(f"after array : {array}, {det_num}")
            break
    return array

def delete_num(array, k):
    for j in range(len(array[k])):
        if array[k][j] != 0:
            break

    for x in range(k+1, len(array)):
        for y in range(len(array[x])-1, j-1, -1):
            if array[k][j] == 0:
                return array
            array[x][y] -= ((array[x][j] / array[k][j]) * array[k][y])
    array[k] = trans_to_one(array[k])
    return array

def trans_to_one(lst):
    global det_num
    for s in range(len(lst)):
        if lst[s] != 0:
            break

    if abs(lst[s]) < 1e-15:
        lst[s] = 0
        return lst

    #print(f"before array : {lst}, {det_num}")
    det_num *= lst[s]

    for t in range(len(lst)-1, s-1, -1):
        lst[t] /= lst[s]
    #print(f"after array : {lst}, {det_num}")
    return lst

def change_to_stair(array):
    for i in range(len(array)-1):
        array = switch_line(array, i)
        array = delete_num(array,i)

    array[len(array)-1] = trans_to_one(array[len(array)-1])
    return array

def change_to_simple(array):
    for i in range(1, len(array)):
        for j in range(len(array[i])):
            if array[i][j] != 0:
                break
        for q in range(i):
            for p in range(len(array[i])-1, j-1, -1):
                array[q][p] -= array[q][j] * array[i][p]
    return array

def gauss(arr):
    array = arr
    global det_num
    det_num = 1
    array = change_to_stair(array)
    array = change_to_simple(array)
    return array

########################################################################################
########################################################################################

def  I(n):
    result = []
    for i in range(n):
        result.append([0 for x in range(n)])
    for i in range(n):
        result[i][i] = 1
    return result

def narray(dim, init_value=1):
    result = []
    for i in range(dim[0]):
        result.append([])

    for i in range(dim[0]):
        for j in range(dim[1]):
            result[i].append(init_value)

    return Matrix(data=result, dim=dim, init_value=init_value)

def arange(start,end,step=1):
    ans = [[]]
    for i in range(start,end,step):
        ans[0].append(i)
    return Matrix(data=ans)


def zeros(dim):
    result = []
    for i in range(dim[0]):
        result.append([])

    for i in range(dim[0]):
        for j in range(dim[1]):
            result[i].append(0)

    return Matrix(data=result, dim=dim)

def zeros_like(matrix):
    return zeros(matrix.dim)

def ones(dim):
    result = []
    for i in range(dim[0]):
        result.append([])

    for i in range(dim[0]):
        for j in range(dim[1]):
            result[i].append(1)

    return Matrix(data=result, dim=dim)

def ones_like(matrix):
    return ones(matrix.dim)

def nrandom(dim):
    result = []
    for i in range(dim[0]):
        result.append([])

    for i in range(dim[0]):
        for j in range(dim[1]):
            ran = random.choice([0, 1])
            result[i].append(ran)

    return Matrix(data=result, dim=dim)

def nrandom_like(matrix):
    return nrandom(matrix.dim)

def concentrate(items, axis=0):
    if axis == 0:
        for x in items:
            if x.dim[0] != items[0].dim[0]:
                return "Error. 不能拼接。"
        
        result = []
        for i in range(items[0].dim[0]):
            result.append([])
        
        for i in range(items[0].dim[0]):
            for j in range(len(items)):
                result[i] = result[i] + items[j].data[i]

        return Matrix(data=result)

    elif axis == 1:
        for x in items:
            if x.dim[1] != items[0].dim[1]:
                return "Error. 不能拼接。"

        result = items[0].data
        for k in range(1, len(items)):
            for i in range(items[k].dim[0]):
                result.append(items[k].data[i])

        return Matrix(data=result)

def vectorize(func):
    def F(matrix):
        result = matrix.data
        for i in range(matrix.dim[0]):
            for j in range(matrix.dim[1]):
                result[i][j] = func(matrix.data[i][j])
        return Matrix(data=result)
    return F


if __name__ == "__main__":
    print("test here")
    pass


A = Matrix(data=[[1,2,3], [4,5,6], [7,8,4]])
print(A.det())
print(A.inverse())
print(A.rank())


test here
15.0
[[ 1.0 0.0 0.0
 0.0 1.0 0.0
 0.0 0.0 1.0
]]
3


In [36]:
def gauss(array):
    global det_num #用于记录行列式的变化
    det_num = 1
    array = change_to_stair(array) #首先化为阶梯型
    array = change_to_simple(array) #然后化为简化阶梯形
    return array


def is_all_zero(lst): #判断某一行是否全为零
    for i in lst:
        if i != 0:
            return False
    return True

def switch_line(array, k): #用于判断某行第k个元素是否为零，若是，则通过行交换将其变为非零
    global det_num
    for i in range(k, len(array)):
        if array[i][0] != 0:
            array[k], array[i] = array[i], array[k] 
            if i != k:
                det_num *= -1
            break
    return array

def delete_num(array, k): #将某列在k行一下的元素全变为零
    #寻找主元
    for j in range(len(array[k])):
        if array[k][j] != 0:
            break

    if array[k][j] == 0:
        return array

    #消去j列k行之下的元素
    for x in range(k+1, len(array)):
        for y in range(len(array[x])-1, j-1, -1):
            array[x][y] -= ((array[x][j] / array[k][j]) * array[k][y])
    array[k] = trans_to_one(array[k])
    return array

def trans_to_one(lst): #将某行主元化为1
    global det_num
    #寻找主元
    for s in range(len(lst)):
        if lst[s] != 0:
            break

    #判断是否此行全为零
    if abs(lst[s]) < 1e-15: #注：用极小数是为了防止浮点数造成的误差
        lst[s] = 0
        return lst

    det_num *= lst[s]

    #主元化为1
    for t in range(len(lst)-1, s-1, -1):
        lst[t] /= lst[s]
    return lst

def change_to_stair(array): #变为阶梯形
    for i in range(len(array)-1):
        array = switch_line(array, i) #先交换行
        array = delete_num(array,i) #再消去主元

    array[len(array)-1] = trans_to_one(array[len(array)-1])
    return array

def change_to_simple(array): #变为简化阶梯形
    #寻找主元
    for i in range(1, len(array)):
        for j in range(len(array[i])):
            if array[i][j] != 0:
                break
    
        #消去每一列主元以上的元素
        for q in range(i):
            for p in range(len(array[i])-1, j-1, -1):
                array[q][p] -= array[q][j] * array[i][p]
    return array

print(gauss([[1,2,3],[4,5,6],[7,8,9]]))


[[1.0, 0.0, -1.0], [0.0, 1.0, 2.0], [0.0, 0.0, 0]]


In [12]:
a = [0 for x in range(100)]
a[0] = 2
a[1] = -2
for i in range(2, 100):
    a[i] = 2 * a[i-1] - 6 * a[i-2]

print(a)

[2, -2, -16, -20, 56, 232, 128, -1136, -3040, 736, 19712, 35008, -48256, -306560, -323584, 1192192, 4325888, 1498624, -22958080, -54907904, 27932672, 385312768, 603029504, -1105817600, -5829812224, -5024718848, 24929435648, 80007184384, 10437754880, -459167596544, -980961722368, 793082134528, 7471934603264, 10185376399360, -24460854820864, -110033968037888, -73302807150592, 513598193926144, 1467013230755840, -147562702045184, -9097204788625408, -17309033364979712, 19965162001793024, 143784524193464320, 167778076376170496, -527150992408444928, -2060970443073912832, -959034931697156096, 10447752795049164800, 26649715180281266176, -9387086409732456448, -178672463901152509952, -301022409343910281216, 469989964719094497280, 2746114385501650681856, 2672288982688734380032, -11132108347632435331072, -38297950591397276942336, -9803251096999941898240, 210181201354383777857536, 479181909290767207104512, -302723389544768252936192, -3480538234834139748499456, -5144736132399669979381760, 10593757144

In [25]:

def test(lst):
    lst = test2(lst)
    return lst
    
def test2(lst):
    global a
    a = 1
    for i in range(len(lst)):
        lst[i] *= 2
        a += 1
    lst = test3(lst)
    return lst

def test3(lst):
    global a
    for i in range(len(lst)):
        lst[i] -= 1
        a += 1
    return lst

lst1 = test([1,2,3,4])
print(lst1)
lst1.append(a)
print(lst1)


[1, 3, 5, 7]
[1, 3, 5, 7, 9]


In [ ]:
##对齐？
    #def __str__(self):
    #    result = "[["
    #    for i in range(self.dim[0]):
    #        for j in range(self.dim[1]):
    #            result = result + " " + str(self.data[i][j])
    #        result = result + "\n"
    #    result = result + "]]"
    #    return result


In [46]:
print(0.00000000000000001)

1e-17
